### 검색 Tool이 필요한 이유
- LLM의 파라미터에 담긴 지식은 최신 사실을 보장하지 않음
- 웹 검색 도구는 외부 자료를 가져오는 역할
- 검색 결과 자체가 틀리거나 악성 지시를 포함할 수 있으므로 신뢰할 수 있는 출처와 날짜를 확인

```
사용자 -> LLM -> 검색 필요? -> 아니요 -> 답변
                                네  -> 검색  Tool -> 결과 확인 -> LLM -> 답변
```

# **1. Tavily**

AI 애플리케이션과 Agent가 웹에서 정보를 검색하고, 검색 결과의 제목/주소/본문 일부를 구조화된 데이터로 받도록 지원하는 검색 API 서비스

In [2]:
import os
from dotenv import load_dotenv

In [4]:
load_dotenv()
OPENAI_API_KEY=bool(os.getenv("OPENAI_API_KEY"))
TAVILY_API_KEY=bool(os.getenv("TAVILY_API_KEY"))
OPENAI_MODEL=bool(os.getenv("OPENAI_MODEL"))

print("OPENAI_API_KEY: ", OPENAI_API_KEY, " | TAVILY_API_KEY: ", TAVILY_API_KEY, " | OPENAI_MODEL", OPENAI_MODEL)

OPENAI_API_KEY:  True  | TAVILY_API_KEY:  True  | OPENAI_MODEL True


In [15]:
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-6-luna")

In [16]:
from langchain_openai import ChatOpenAI

In [17]:
def make_llm():
    # API 키가 없으면 오류 메시지를 출력하고 None 반환
    if not OPENAI_API_KEY:
        print("KEY ERROR!")
        return None

    # OpenAI LLM 객체 생성 및 반환
    return ChatOpenAI(
        model=MODEL_NAME,        # 사용할 모델 이름
        reasoning_effort="none", # 추론 노력 수준 설정 (추론 최소화)
        timeout=30,              # API 응답 대기 시간 (최대 30초)
        max_retries=2,           # API 요청 실패 시 최대 재시도 횟수
        temperature=0            # 응답의 무작위성 설정 (0: 일관성 높은 응답)
    )

In [18]:
llm = make_llm()

In [19]:
from tavily import TavilyClient  # pip install tavily

In [21]:
# Tavily API 키가 존재하는 경우
if TAVILY_API_KEY:
    # Tavily 검색 클라이언트 생성
    tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

    # 웹 검색 실행 (최대 3개 결과 반환)
    raw_search = tavily_client.search(query="랭그래프는 뭐야?", max_results=3)

    # 검색 결과의 최상위 키 출력
    print("최상위 키: ", list(raw_search.keys()))

    # 검색 결과를 하나씩 순회 (인덱스는 1부터 시작)
    for idx, item in enumerate(raw_search.get("results", []), start=1):
        print(f"\n[{idx}]제목: ", item.get("title", ""))               # 검색 결과 제목
        print("주소: ", item.get("url", ""))                 # 검색 결과 URL
        print("요약: ", str(item.get("content", ""))[:200])  # 검색 내용의 앞 200자 출력
else:
    # API 키가 없는 경우
    print("KEY ERROR!!")

최상위 키:  ['query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id']

[1]제목:  LangGraph란 무엇이고 언제 쓰면 좋을까?
주소:  https://velog.io/@ohback/LangGraph
요약:  여기서 등장하는 게 LangGraph다. 이름 그대로 그래프(노드/엣지)로 LLM 애플리케이션의 흐름을 설계한다.  
 LangChain이 블록(모델·프롬프트·툴)을 잘 조립하게 해주는 공구상자라면, LangGraph는 그 블록들이 언제, 어떤 조건으로, 어떤 상태를 들고 움직일지를 정밀하게 오케스트레이션하는 조감도 + 신호등에 가깝다.

  

## 그래

[2]제목:  LangGraph란 무엇인가요? | IBM
주소:  https://www.ibm.com/kr-ko/think/topics/langgraph
요약:  # LangGraph란 무엇인가요?

Bryan Clark 

Senior AI Advocate

## LangGraph란 무엇인가요?

LangChain에서 만든 LangGraph는 복잡한 생성형 AI 에이전트 워크플로를 구축, 배포, 관리하도록 설계된 오픈 소스 AI 에이전트 프레임워크입니다. 이는 사용자가 확장가능한 방식으로 대규모 언어 모델(LLM)

[3]제목:  꿈 많은 사람의 이야기
주소:  https://lsjsj92.tistory.com/696
요약:  ## LangGraph란?

LangGraph는 LLM을 기반으로 상태를 유지하고(Stateful), 여러 단계에 걸쳐 작동하는(multi-step) 에이전트를 구축하기 위한 라이브러리입니다. 기존의 LangChain이 LLM과 외부 도구를 '연쇄적으로(chaining)' 연결하는 데 중점을 두었다면, LangGraph는 이러한 연결 구조를 순환(cycle
